In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.sparse import hstack
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import f1_score, accuracy_score

from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, MaxPooling1D, \
                                    GlobalMaxPooling1D, GlobalAveragePooling1D, \
                                    Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

In [ ]:
VOCAB_SIZE = 15000
MAX_LENGTH = 50
EMBEDDING_DIM = 32

LEARNING_RATE = 5e-4
BATCH_SIZE = 64
EPOCHS = 10

RANDOM_STATE = 42
STACKING = True

In [ ]:
def clean_text(text):
    # remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    # remove @mentions
    text = re.sub(r'@\w+', '', text)
    ## remove #topics
    #text = re.sub(r'#\w+', '', text)
    # remove non-alphabetical characters but keep white spaces
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    # lower case text
    text = text.lower()
    return ' '.join(text.split())

In [ ]:
def display_confusion_matrix(y_true, y_pred, model_name=''):

    cm = confusion_matrix(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='weighted')
    
    plt.figure(figsize=(7,6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Not Disaster', 'Disaster'],
                yticklabels=['Not Disaster', 'Disaster'])
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(f'Confusion Matrix {model_name}\nF1 Score (weighted): {f1:.3f}', fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
df = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')
df_test = pd.read_csv('/kaggle/input/nlp-getting-started/test.csv')

df

In [ ]:
df['text_clean'] = df['text'].apply(clean_text)
df_test['text_clean'] = df_test['text'].apply(clean_text)

In [ ]:
text, text_clean = df[df["text"].str.contains("#")].sample()[["text", "text_clean"]].values[0]
print("Raw text:\n", text)
print("Clean text:\n", text_clean)

In [ ]:
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(df["text_clean"])

X1 = tokenizer.texts_to_sequences(df["text_clean"])
X1 = pad_sequences(X1, maxlen=MAX_LENGTH, padding='post')

X1_test = tokenizer.texts_to_sequences(df_test["text_clean"])
X1_test = pad_sequences(X1_test, maxlen=MAX_LENGTH, padding='post')

y = df["target"].values

print(X1.shape)
print(X1_test.shape)

In [ ]:
X1_train, X1_val, y_train, y_val = train_test_split(X1, y, 
                                                  test_size=0.2, 
                                                  stratify=y, 
                                                  random_state=RANDOM_STATE)

In [ ]:
model1 = Sequential([
    Embedding(VOCAB_SIZE, EMBEDDING_DIM),
    Conv1D(32, kernel_size=5, activation='relu'),
    GlobalMaxPooling1D(),
    Dropout(0.5),
    Dense(64, activation="relu"),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
], name="ConvEmbedding")

model1.build(input_shape=(BATCH_SIZE, MAX_LENGTH))
model1.summary()

In [ ]:
model1.compile(loss="binary_crossentropy", 
              optimizer=Adam(LEARNING_RATE), 
              metrics=["accuracy"])

In [ ]:
early_stopping = EarlyStopping(monitor='val_loss', patience=3)
model_checkpoint = ModelCheckpoint("best_model.keras", monitor='val_accuracy', save_best_only=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=1, min_lr=1e-4)

In [ ]:
fitting = model1.fit(X1_train, y_train, 
                    validation_data=(X1_val, y_val),  
                    batch_size=BATCH_SIZE,
                    epochs=EPOCHS, 
                    callbacks=[early_stopping, model_checkpoint, reduce_lr])

model1.load_weights("best_model.keras")

In [ ]:
epoch_counts = range(1, early_stopping.stopped_epoch+2)
plt.plot(epoch_counts, fitting.history["loss"])
plt.plot(epoch_counts, fitting.history["val_loss"])
plt.legend(["Training", "Validation"])
plt.title("Loss")
plt.show()

In [ ]:
plt.plot(epoch_counts, fitting.history["accuracy"])
plt.plot(epoch_counts, fitting.history["val_accuracy"])
plt.legend(["Training", "Validation"])
plt.title("Accuracy")
plt.show()

In [ ]:
y1_proba = model1.predict(X1_val).ravel()
y1_pred = (y1_proba >= 0.5).astype(int)

print(y1_proba)
print(y1_pred)

In [ ]:
print(classification_report(y_val, y1_pred, target_names=['Not Disaster', 'Disaster']))

In [ ]:
display_confusion_matrix(y_val, y1_pred, model_name="ConvEmbedding")

In [ ]:
df["length"] = df["text"].apply(len)
df_test["length"] = df_test["text"].apply(len)

df[["text", "length"]]

In [ ]:
vectorizer = TfidfVectorizer(stop_words='english', max_features=VOCAB_SIZE)
X_tfidf = vectorizer.fit_transform(df["text_clean"])
X_test_tfidf = vectorizer.transform(df_test["text_clean"])

df_tfidf = pd.DataFrame(X_tfidf.toarray(), columns=vectorizer.get_feature_names_out())
print(df.loc[0, "text_clean"])
print(df_tfidf.loc[0, df_tfidf.loc[0] != 0])
df_tfidf

In [ ]:
X2 = hstack([X_tfidf, df[["length"]].values])
X2_test = hstack([X_test_tfidf, df_test[["length"]].values])

print(X2.shape)
print(y.shape)

In [ ]:
X2_train, X2_val, y_train, y_val = train_test_split(X2, y, 
                                                  stratify=y, 
                                                  test_size=0.2, 
                                                  random_state=RANDOM_STATE)

In [ ]:
model2 = LogisticRegression(max_iter=500, random_state=RANDOM_STATE)

model2.fit(X2_train, y_train)

In [ ]:
y2_proba = model2.predict_proba(X2_val)[:,1]
y2_pred = (y2_proba >= 0.5).astype(int)

print(y2_proba)
print(y2_pred)

In [ ]:
print(classification_report(y_val, y2_pred, target_names=['Not Disaster', 'Disaster']))

In [ ]:
display_confusion_matrix(y_val, y2_pred, model_name="TF-IDF")

In [ ]:
a = 0.5

y_proba = a * y1_proba + (1 - a) * y2_proba
y_pred = (y_proba >= 0.5).astype(int)

print(classification_report(y_val, y_pred, target_names=['Not Disaster', 'Disaster']))

In [ ]:
display_confusion_matrix(y_val, y_pred, model_name="Aggregated Model")

In [ ]:
output1 = model1.predict(X1_train).ravel()
output2 = model2.predict_proba(X2_train)[:,1]
outputs_train = np.column_stack([output1, output2])
outputs_val = np.column_stack([y1_proba, y2_proba])

meta_learner = LogisticRegression(max_iter=500, random_state=RANDOM_STATE)
meta_learner.fit(outputs_train, y_train)

y_proba = meta_learner.predict_proba(outputs_val)[:,1]
y_pred = (y_proba >= 0.5).astype(int)

print(classification_report(y_val, y_pred, target_names=['Not Disaster', 'Disaster']))

In [ ]:
display_confusion_matrix(y_val, y_pred, model_name="Stacked Model")

In [ ]:
y1_proba = model1.predict(X1_test).ravel()
y2_proba = model2.predict_proba(X2_test)[:,1]

if STACKING:
    outputs = np.column_stack([y1_proba, y2_proba])
    y_proba = meta_learner.predict_proba(outputs)[:,1]
else:
    y_proba = a * y1_proba + (1 - a) * y2_proba

y_pred = (y_proba >= 0.5).astype(int)

print(y_proba)
print(y_pred)

In [ ]:
np.unique(y_pred, return_counts=True)

In [ ]:
df_submission = pd.read_csv('/kaggle/input/nlp-getting-started/sample_submission.csv')
df_submission["target"] = y_pred

df_submission

In [ ]:
df_submission.to_csv("submission.csv", index=False)